# Phase 9 — Multi-seed runs (Tier 2 + Tier 3, N=10)
Runs seeds 42, 123, 7 for DSPy and QLoRA on Tier 2 and Tier 3. This is the biggest GPU chunk in the whole roadmap — budget MULTIPLE Colab sessions, not one sitting. Each of the 4 stages below is a natural stopping point.

**Before running:** Runtime -> Change runtime type -> T4 GPU.

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
!nvidia-smi

**STOP: must show 0MiB used before continuing.**

## Setup (repeat this after every restart)

In [ ]:
import shutil, os
os.chdir("/content")
if os.path.exists("agentic-prompt-vs-finetune"):
    shutil.rmtree("agentic-prompt-vs-finetune")
!git clone https://github.com/nive62tech/agentic-prompt-vs-finetune.git
%cd agentic-prompt-vs-finetune
!pip install -q transformers accelerate bitsandbytes peft datasets dspy-ai optuna

In [ ]:
from huggingface_hub import login
login()

---
## STAGE A — Tier 2 DSPy, 3 seeds
Load the model once, run all 3 seeds in this session (cache-cleared between each).

In [ ]:
import sys, json, gc, torch
sys.path.insert(0, ".")
from envs.agent_harness import load_model
from envs.dspy_lm import LocalLlamaLM
from envs.training_data import sample_tier2_training
from tasks.tier2 import TIER2_HELDOUT
import dspy_optimize_tier2

model, tok = load_model("meta-llama/Llama-3.1-8B-Instruct")
lm = LocalLlamaLM(model, tok)
print("Model ready.")

In [ ]:
SEEDS = [42, 123, 7]

for seed in SEEDS:
    gc.collect(); torch.cuda.empty_cache()
    print(f"\n=== Tier 2 DSPy, seed={seed} ===")
    train = sample_tier2_training(10, seed=seed)
    optimized = dspy_optimize_tier2.optimize(lm, train, seed=seed)
    results = dspy_optimize_tier2.evaluate_program(optimized, TIER2_HELDOUT)
    rate = sum(r["grade"]["success"] for r in results) / len(results)
    print(f"Tier 2 DSPy seed={seed}: {rate:.1%}")
    with open(f"results/tier2_dspy_n10_seed{seed}_results.json", "w") as f:
        json.dump(results, f, indent=2)

In [ ]:
from google.colab import files
for seed in SEEDS:
    files.download(f"results/tier2_dspy_n10_seed{seed}_results.json")

**Download the 3 files above before continuing.** Move them into `results/` on your laptop and push. Then RESTART the runtime before Stage B (Disconnect and delete runtime, not just Restart session).

---
## STAGE B — Tier 3 DSPy (guarded), 3 seeds
Re-run the Setup cells above first, then this.

In [ ]:
import sys, json, gc, torch
sys.path.insert(0, ".")
from envs.agent_harness import load_model
from envs.dspy_lm import LocalLlamaLM
from envs.training_data import sample_tier3_training
from tasks.tier3 import TIER3_HELDOUT
import dspy_optimize_tier3

model, tok = load_model("meta-llama/Llama-3.1-8B-Instruct")
lm = LocalLlamaLM(model, tok)
print("Model ready.")

In [ ]:
SEEDS = [42, 123, 7]

for seed in SEEDS:
    gc.collect(); torch.cuda.empty_cache()
    print(f"\n=== Tier 3 DSPy (guarded), seed={seed} ===")
    train = sample_tier3_training(10, seed=seed)
    optimized = dspy_optimize_tier3.optimize(lm, train, guarded=True, seed=seed)
    results = dspy_optimize_tier3.evaluate_program(optimized, TIER3_HELDOUT)
    rate = sum(r["grade"]["success"] for r in results) / len(results)
    print(f"Tier 3 DSPy (guarded) seed={seed}: {rate:.1%}")
    with open(f"results/tier3_dspy_guarded_n10_seed{seed}_results.json", "w") as f:
        json.dump(results, f, indent=2)

In [ ]:
from google.colab import files
for seed in SEEDS:
    files.download(f"results/tier3_dspy_guarded_n10_seed{seed}_results.json")

**Download, push, then RESTART before Stage C.**

---
## STAGE C — Tier 2 QLoRA, 3 seeds
Re-run Setup first. Each seed is a fresh subprocess call — safer against memory buildup than the DSPy stages.

In [ ]:
for seed in [42, 123, 7]:
    print(f"\n=== Tier 2 QLoRA, seed={seed} ===")
    !python qlora_finetune_tier2.py --n 10 --seed {seed}

In [ ]:
import torch, json, gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from envs.agent_harness import run_agent
from envs.tools import TOOL_SCHEMAS, call_tool
from tasks.tier2 import TIER2_HELDOUT
from grader import grade_task

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4")

for seed in [42, 123, 7]:
    print(f"\n=== Evaluating Tier 2 QLoRA, seed={seed} ===")
    base_model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.1-8B-Instruct", quantization_config=bnb_config, device_map={"": 0})
    ft_model = PeftModel.from_pretrained(base_model, f"adapters/tier2_n10_seed{seed}")
    ft_tok = AutoTokenizer.from_pretrained(f"adapters/tier2_n10_seed{seed}")

    results = []
    for task in TIER2_HELDOUT:
        tool_calls, final_text = run_agent(ft_model, ft_tok, task["prompt"], TOOL_SCHEMAS, call_tool, max_turns=6)
        grade = grade_task(task, tier=2, tool_calls=tool_calls, final_text=final_text)
        results.append({"id": task["id"], "prompt": task["prompt"], "tool_calls": tool_calls, "final_text": final_text, "grade": grade})
    rate = sum(r["grade"]["success"] for r in results) / len(results)
    print(f"Tier 2 QLoRA seed={seed}: {rate:.1%}")
    with open(f"results/tier2_qlora_n10_seed{seed}_results.json", "w") as f:
        json.dump(results, f, indent=2)

    del base_model, ft_model
    gc.collect(); torch.cuda.empty_cache()

In [ ]:
from google.colab import files
for seed in [42, 123, 7]:
    files.download(f"results/tier2_qlora_n10_seed{seed}_results.json")

**Download, push, then RESTART before Stage D.**

---
## STAGE D — Tier 3 QLoRA, 3 seeds
Re-run Setup first.

In [ ]:
for seed in [42, 123, 7]:
    print(f"\n=== Tier 3 QLoRA, seed={seed} ===")
    !python qlora_finetune_tier3.py --n 10 --seed {seed}

In [ ]:
import torch, json, gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from envs.agent_harness import run_agent
from envs.tools import TOOL_SCHEMAS, call_tool
from tasks.tier3 import TIER3_HELDOUT
from grader import grade_task

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4")

for seed in [42, 123, 7]:
    print(f"\n=== Evaluating Tier 3 QLoRA, seed={seed} ===")
    base_model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.1-8B-Instruct", quantization_config=bnb_config, device_map={"": 0})
    ft_model = PeftModel.from_pretrained(base_model, f"adapters/tier3_n10_seed{seed}")
    ft_tok = AutoTokenizer.from_pretrained(f"adapters/tier3_n10_seed{seed}")

    results = []
    for task in TIER3_HELDOUT:
        tool_calls, final_text = run_agent(ft_model, ft_tok, task["prompt"], TOOL_SCHEMAS, call_tool, max_turns=6)
        grade = grade_task(task, tier=3, tool_calls=tool_calls, final_text=final_text)
        results.append({"id": task["id"], "prompt": task["prompt"], "tool_calls": tool_calls, "final_text": final_text, "grade": grade})
    rate = sum(r["grade"]["success"] for r in results) / len(results)
    print(f"Tier 3 QLoRA seed={seed}: {rate:.1%}")
    with open(f"results/tier3_qlora_n10_seed{seed}_results.json", "w") as f:
        json.dump(results, f, indent=2)

    del base_model, ft_model
    gc.collect(); torch.cuda.empty_cache()

In [ ]:
from google.colab import files
for seed in [42, 123, 7]:
    files.download(f"results/tier3_qlora_n10_seed{seed}_results.json")

## Done with N=10 x 3 seeds. Aggregate locally (no GPU needed):
```powershell
python aggregate_seeds.py results/tier2_dspy_n10_seed42_results.json results/tier2_dspy_n10_seed123_results.json results/tier2_dspy_n10_seed7_results.json
python aggregate_seeds.py results/tier2_qlora_n10_seed42_results.json results/tier2_qlora_n10_seed123_results.json results/tier2_qlora_n10_seed7_results.json
python aggregate_seeds.py results/tier3_dspy_guarded_n10_seed42_results.json results/tier3_dspy_guarded_n10_seed123_results.json results/tier3_dspy_guarded_n10_seed7_results.json
python aggregate_seeds.py results/tier3_qlora_n10_seed42_results.json results/tier3_qlora_n10_seed123_results.json results/tier3_qlora_n10_seed7_results.json
```

## N=50 fill-ins (stretch goal, separate future session)
Same code, same notebook structure — just change `10` to `50` in the `sample_tierN_training(...)` calls and the `--n` flags. Tier 2's pool (~93 examples) and Tier 3's pool (~66 examples) both support N=50.